# NOAA GHCN-D — Relevamiento reproducible de Argentina

**Objetivo:** medir estaciones, disponibilidad de variables y tamaños **reales** de objetos del bucket público `noaa-ghcn-pds` antes de implementar la ingesta. Este notebook **no escribe en AWS**.

Fuente de partida: [documentación GHCN-D en AWS](https://github.com/awslabs/open-data-docs/tree/main/docs/noaa/noaa-ghcn) y [explorador del bucket vinculado](https://noaa-ghcn-pds.s3.amazonaws.com/index.html).

**Dos niveles de medición:** (1) inventario y tamaños S3 reales, sin descargar observaciones; (2) medición exacta de filas/bytes CSV que pasarían los filtros, opcional, leyendo los archivos de estaciones. Los metadatos del inventario indican **primer y último año**, no continuidad diaria.

## 1. Parámetros y dependencias

Cambiar `PAISES`, `FECHA_DESDE`, `FECHA_HASTA` y `VARIABLES` para otros escenarios. El prefijo S3 por estación se valida con el listado real del bucket; no se supone que los archivos estén disponibles sin comprobarlo.

In [37]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import csv
import gzip
import io
import shutil
from pathlib import Path

import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

### Parámetros

In [39]:
BUCKET = "noaa-ghcn-pds"
PAISES = ["NZ"]
FECHA_DESDE = "2016-01-01"
FECHA_HASTA = "2026-12-31"
VARIABLES = ["PRCP", "TMAX", "TMIN", "TAVG", "SNOW"]
PREFIJO_ESTACIONES = "csv.gz/by_station/"  # verificar con el explorador del bucket https://noaa-ghcn-pds.s3.amazonaws.com/index.html

In [40]:
CACHE = Path("datos_auxiliares_noaa")
CACHE.mkdir(exist_ok=True)

inicio = pd.Timestamp(FECHA_DESDE)
fin = pd.Timestamp(FECHA_HASTA)

assert inicio <= fin
assert PAISES and VARIABLES

s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))
print("Configuración:", PAISES, FECHA_DESDE, FECHA_HASTA, VARIABLES)

Configuración: ['NZ'] 2016-01-01 2026-12-31 ['PRCP', 'TMAX', 'TMIN', 'TAVG', 'SNOW']


## 2. Descargar únicamente los metadatos del bucket

Se guardan copias locales para evitar repetir las transferencias durante el análisis. Si se requiere actualizar, cambiar `ACTUALIZAR = True`.

In [41]:
ACTUALIZAR = False
archivos = ["ghcnd-stations.txt", "ghcnd-inventory.txt", "ghcnd-countries.txt"]
for nombre in archivos:
    destino = CACHE / nombre
    if ACTUALIZAR or not destino.exists():
        s3.download_file(BUCKET, nombre, str(destino))
    print(f"{nombre}: {destino.stat().st_size:,} bytes locales")

ghcnd-stations.txt: 11,441,526 bytes locales
ghcnd-inventory.txt: 36,400,398 bytes locales
ghcnd-countries.txt: 3,659 bytes locales


## 3. Interpretar estaciones e inventario (campos de ancho fijo)

En el inventario, `FIRSTYEAR` y `LASTYEAR` indican los extremos de disponibilidad del elemento, **no que todos los años o días intermedios tengan datos**.

In [42]:
estaciones = pd.read_fwf(
    CACHE / "ghcnd-stations.txt", header=None,
    colspecs=[(0,11),(12,20),(21,30),(31,37),(38,40),(41,71)],
    names=["ID","LATITUD","LONGITUD","ELEVACION","ESTADO","NOMBRE"],
    dtype={"ID":str,"ESTADO":str,"NOMBRE":str}
)
inventario = pd.read_fwf(
    CACHE / "ghcnd-inventory.txt", header=None,
    colspecs=[(0,11),(12,20),(21,30),(31,35),(36,40),(41,45)],
    names=["ID","LATITUD","LONGITUD","ELEMENT","FIRSTYEAR","LASTYEAR"],
    dtype={"ID":str,"ELEMENT":str}
)

for tabla in (estaciones,inventario):
    tabla["ID"] = tabla["ID"].str.strip()

inventario["ELEMENT"] = inventario["ELEMENT"].str.strip()

sel_estaciones = estaciones[estaciones.ID.str[:2].isin(PAISES)].copy()
sel_inventario = inventario[
    inventario.ID.isin(sel_estaciones.ID) &
    inventario.ELEMENT.isin(VARIABLES) &
    (inventario.FIRSTYEAR <= fin.year) &
    (inventario.LASTYEAR >= inicio.year)
].copy()

print("Estaciones en países elegidos:", sel_estaciones.ID.nunique())
print("Estaciones candidatas (alguna variable y solapamiento temporal):", sel_inventario.ID.nunique())
print("Entradas estación-variable candidatas:", len(sel_inventario))

resumen_variables = (sel_inventario.groupby("ELEMENT").agg(
    estaciones=("ID","nunique"), primer_anio=("FIRSTYEAR","min"), ultimo_anio=("LASTYEAR","max")
).reindex(VARIABLES))

display(resumen_variables)
display(sel_estaciones.head())

Estaciones en países elegidos: 15
Estaciones candidatas (alguna variable y solapamiento temporal): 14
Entradas estación-variable candidatas: 53


,estaciones,primer_anio,ultimo_anio
ELEMENT,,,
PRCP,13.0,1960.0,2026.0
TMAX,13.0,1940.0,2026.0
TMIN,13.0,1940.0,2026.0
TAVG,14.0,1954.0,2026.0
SNOW,NaN,NaN,NaN


,ID,LATITUD,LONGITUD,ELEVACION,ESTADO,NOMBRE
48106,NZ000093012,-35.100,173.267,54.0,NaN,KAITAIA
48107,NZ000093292,-38.650,177.983,5.0,NaN,GISBORNE AERODROME
48108,NZ000093417,-40.900,174.983,7.0,NaN,PARAPARAUMU AWS
48109,NZ000093844,-46.417,168.333,2.0,NaN,INVERCARGILL AIRPOR
48110,NZ000093994,-29.250,-177.917,49.0,NaN,RAOUL ISL/KERMADEC


## 4. Cobertura potencial y distribución

Los conteos siguientes se basan en los intervalos publicados en el inventario; no prueban completitud diaria ni disponibilidad de cada mes.

In [43]:
anios = range(inicio.year, fin.year + 1)

filas = []
for anio in anios:
    cobertura = sel_inventario[(sel_inventario.FIRSTYEAR <= anio) & (sel_inventario.LASTYEAR >= anio)]
    for var in VARIABLES:
        filas.append({"anio":anio,"variable":var,"estaciones_potenciales":cobertura.loc[cobertura.ELEMENT.eq(var),"ID"].nunique()})

cobertura_anual = pd.DataFrame(filas).pivot(index="anio",columns="variable",values="estaciones_potenciales")
display(cobertura_anual.reindex(columns=VARIABLES))

matriz_variables = (sel_inventario.assign(disponible=True)
                    .pivot_table(index="ID",columns="ELEMENT",values="disponible",aggfunc="max",fill_value=False))

print("Estaciones potenciales con PRCP:", int(matriz_variables.get("PRCP",pd.Series(dtype=bool)).sum()))
print("Estaciones potenciales con todas las variables solicitadas:", int(matriz_variables.reindex(columns=VARIABLES,fill_value=False).all(axis=1).sum()))

variable,PRCP,TMAX,TMIN,TAVG,SNOW
anio,,,,,
2016,12,13,13,13,0
2017,12,13,13,13,0
2018,12,13,13,13,0
2019,12,13,13,13,0
2020,12,13,13,13,0
2021,12,13,13,13,0
2022,12,13,13,13,0
2023,12,13,13,13,0
2024,12,13,13,13,0


Estaciones potenciales con PRCP: 13
Estaciones potenciales con todas las variables solicitadas: 0


## 5. Medir tamaños de archivos de estaciones en S3 (sin descargarlos)

Se consulta el listado de objetos bajo el prefijo de archivos comprimidos por estación. Cada `Size` es el tamaño real almacenado en S3 (bytes comprimidos), **no** el tamaño tras filtrar fechas/variables. Se informa si faltan archivos esperados.

Si no se hallan objetos, inspeccionar el explorador y ajustar `PREFIJO_ESTACIONES` antes de continuar.

In [44]:
candidatas = set(sel_inventario.ID)
objetos = []
paginador = s3.get_paginator("list_objects_v2")
for pais in PAISES:
    prefijo = PREFIJO_ESTACIONES + pais
    for pagina in paginador.paginate(Bucket=BUCKET, Prefix=prefijo):
        for objeto in pagina.get("Contents", []):
            key = objeto["Key"]
            identificador = Path(key).name.removesuffix(".csv.gz")
            if identificador in candidatas and key.endswith(".csv.gz"):
                objetos.append({"ID":identificador,"key":key,"bytes_comprimidos":objeto["Size"]})
tamanios = pd.DataFrame(objetos,columns=["ID","key","bytes_comprimidos"]).drop_duplicates("key")
print("Archivos encontrados:",len(tamanios),"de",len(candidatas),"estaciones candidatas")
print("Estaciones sin archivo encontrado:",len(candidatas-set(tamanios.ID)))
print("Volumen total real comprimido: %.3f GiB" % (tamanios.bytes_comprimidos.sum()/1024**3))
display(tamanios.sort_values("bytes_comprimidos",ascending=False).head(15))

if tamanios.empty:
    print("ATENCIÓN: revisar PREFIJO_ESTACIONES en el explorador S3; no inferir volumen cero.")

Archivos encontrados: 14 de 14 estaciones candidatas
Estaciones sin archivo encontrado: 0
Volumen total real comprimido: 0.003 GiB


,ID,key,bytes_comprimidos
3,NZ000093844,csv.gz/by_station/NZ000093844.csv.gz,407639
4,NZ000093994,csv.gz/by_station/NZ000093994.csv.gz,386812
5,NZ000933090,csv.gz/by_station/NZ000933090.csv.gz,347658
8,NZ000939450,csv.gz/by_station/NZ000939450.csv.gz,316099
1,NZ000093292,csv.gz/by_station/NZ000093292.csv.gz,293811
12,NZM00093781,csv.gz/by_station/NZM00093781.csv.gz,291477
6,NZ000936150,csv.gz/by_station/NZ000936150.csv.gz,286893
2,NZ000093417,csv.gz/by_station/NZ000093417.csv.gz,278928
0,NZ000093012,csv.gz/by_station/NZ000093012.csv.gz,211010
7,NZ000937470,csv.gz/by_station/NZ000937470.csv.gz,185924


## 6. Medición efectiva del volumen y calidad de PRCP

Se leen **una sola vez** los CSV comprimidos por estación, directamente desde S3 y sin guardarlos localmente. Se mide el subconjunto parametrizado y se conservan en memoria únicamente las observaciones de precipitación (`PRCP`) necesarias para validar la cobertura. El tamaño CSV medido corresponde a filas seleccionadas, sin cabeceras; no equivale al tamaño comprimido de RAW.

In [45]:
# Lectura única para volumen y evaluación de precipitaciones
resultados = []
registros_prcp = []
desde = inicio.strftime("%Y%m%d")
hasta = fin.strftime("%Y%m%d")

for idx, obj in enumerate(tamanios.sort_values("ID").itertuples(index=False), 1):
    respuesta = s3.get_object(Bucket=BUCKET, Key=obj.key)
    contados = 0
    bytes_csv = 0
    por_variable = {v: 0 for v in VARIABLES}
    try:
        with gzip.GzipFile(fileobj=respuesta["Body"]) as descomprimido:
            with io.TextIOWrapper(descomprimido, encoding="utf-8", newline="") as entrada:
                for linea in entrada:
                    partes = next(csv.reader([linea]))
                    if partes and partes[0].strip().upper() == "ID":
                        continue
                    if len(partes) < 6:
                        raise ValueError(f"Formato CSV inesperado: {obj.key}")
                    fecha = partes[1].strip().replace("-", "")
                    variable = partes[2].strip()
                    if desde <= fecha <= hasta and variable in por_variable:
                        contados += 1
                        por_variable[variable] += 1
                        bytes_csv += len(linea.rstrip("\r\n").encode("utf-8")) + 1
                        if variable == "PRCP":
                            registros_prcp.append({
                                "ID": obj.ID,
                                "FECHA": fecha,
                                "VALOR": partes[3].strip(),
                                "Q_FLAG": partes[5].strip()
                            })
    finally:
        respuesta["Body"].close()
    resultados.append({"ID": obj.ID, "filas_filtradas": contados,
                       "bytes_csv_filtrado": bytes_csv, **por_variable})
    if idx % 25 == 0:
        print(f"Procesadas {idx}/{len(tamanios)} estaciones")

medicion = pd.DataFrame(resultados)
prcp = pd.DataFrame(registros_prcp)
prcp["FECHA"] = pd.to_datetime(prcp["FECHA"], format="%Y%m%d")
prcp["VALOR"] = pd.to_numeric(prcp["VALOR"], errors="coerce")
prcp["ANIO"] = prcp["FECHA"].dt.year
prcp["MES"] = prcp["FECHA"].dt.month
prcp["valido"] = prcp["Q_FLAG"].eq("") & prcp["VALOR"].notna() & prcp["VALOR"].ge(0)

print("Estaciones con archivos:", len(medicion))
print("Filas seleccionadas:", medicion.filas_filtradas.sum())
print("Bytes CSV seleccionados (sin cabeceras):", medicion.bytes_csv_filtrado.sum())
display(medicion[VARIABLES].sum().rename("filas por variable").to_frame())

Estaciones con archivos: 14
Filas seleccionadas: 155892
Bytes CSV seleccionados (sin cabeceras): 5419737


,filas por variable
PRCP,39366
TMAX,37207
TMIN,31250
TAVG,48069
SNOW,0


## 7. Interpretación de volúmenes

- **Inventario:** períodos potenciales, sin garantía de continuidad.
- **Listado S3:** bytes comprimidos de los archivos completos por estación.
- **Medición CSV:** bytes reales de las filas filtradas, todavía sin comprimir.
- La calidad y completitud de las precipitaciones se analiza a continuación; se preservan todos los registros en el cálculo de volumen RAW.

## 8. Viabilidad de acumulados mensuales de precipitación

Evaluación única y final para el caso de uso: conteos de precipitación igual a cero, positiva y observaciones con indicador de calidad; cobertura mensual **con registros válidos** para 2016–2025. El año 2026 se informa separadamente porque está incompleto. La cobertura se calcula sobre todos los días calendario del mes; meses sin registros cuentan como incompletos. Para el ranking también se considera el intervalo efectivo entre el primer y último mes observado por estación, sin ocultar los meses sin registros dentro de ese intervalo.

El umbral de 90% es un criterio técnico explícito para evaluar la posibilidad de calcular acumulados mensuales; no se eliminan datos originales.

In [46]:
UMBRAL_COBERTURA = 0.90
ANIO_HISTORICO_FIN = 2025

# La validación usa las observaciones PRCP ya obtenidas en la sección 6.
calidad = pd.DataFrame({
    "tipo": ["PRCP = 0 válido", "PRCP > 0 válido", "PRCP no válido"],
    "registros": [int((prcp.valido & prcp.VALOR.eq(0)).sum()),
                  int((prcp.valido & prcp.VALOR.gt(0)).sum()),
                  int((~prcp.valido).sum())]
})
display(calidad)
print("Indicadores Q_FLAG en PRCP:")
display(prcp.groupby("Q_FLAG", dropna=False).size().rename("registros").to_frame())

# Se evalúan solo meses completos del período histórico aprobado.
historico_inicio = inicio
historico_fin = min(fin, pd.Timestamp(f"{ANIO_HISTORICO_FIN}-12-31"))
prcp_historico = prcp.loc[
    prcp.valido & prcp.FECHA.between(historico_inicio, historico_fin)
].copy()
print("Estaciones con PRCP válida en 2016–2025:", prcp_historico.ID.nunique())
print("Registros PRCP válidos en 2016–2025:", len(prcp_historico))

,tipo,registros
0,PRCP = 0 válido,19010
1,PRCP > 0 válido,20355
2,PRCP no válido,1


Indicadores Q_FLAG en PRCP:


,registros
Q_FLAG,
,39365
O,1


Estaciones con PRCP válida en 2016–2025: 13
Registros PRCP válidos en 2016–2025: 38595


In [47]:
# Actividad anual real; se presenta 2026 por separado, sin atribuirle completitud.
resumen_anual = (
    prcp.groupby("ANIO")
    .agg(registros=("FECHA", "size"), estaciones=("ID", "nunique"),
         registros_validos=("valido", "sum"))
    .reset_index()
)
display(resumen_anual)

,ANIO,registros,estaciones,registros_validos
0,2016,4267,12,4267
1,2017,4307,12,4307
2,2018,4141,12,4141
3,2019,3975,12,3975
4,2020,4061,12,4061
5,2021,3980,12,3980
6,2022,4184,12,4184
7,2023,4298,12,4297
8,2024,4193,12,4193
9,2025,1190,13,1190


In [48]:
# Cobertura mensual: fechas válidas únicas / días calendario.
meses = pd.period_range(historico_inicio, historico_fin, freq="M")
ids = sorted(prcp_historico.ID.unique())
indice = pd.MultiIndex.from_product([ids, meses], names=["ID", "PERIODO"])
cobertura_mensual = indice.to_frame(index=False)

observaciones = (
    prcp_historico.assign(PERIODO=prcp_historico.FECHA.dt.to_period("M"))
    .groupby(["ID", "PERIODO"])["FECHA"].nunique()
    .rename("dias_validos")
    .reset_index()
)
cobertura_mensual = cobertura_mensual.merge(
    observaciones, on=["ID", "PERIODO"], how="left"
)
cobertura_mensual["dias_validos"] = cobertura_mensual.dias_validos.fillna(0).astype(int)
cobertura_mensual["dias_mes"] = cobertura_mensual.PERIODO.dt.days_in_month
cobertura_mensual["cobertura"] = cobertura_mensual.dias_validos / cobertura_mensual.dias_mes
cobertura_mensual["suficiente"] = cobertura_mensual.cobertura >= UMBRAL_COBERTURA
cobertura_mensual["ANIO"] = cobertura_mensual.PERIODO.dt.year

# Identificamos meses activos sin suponer cobertura fuera de los extremos observados.
extremos = prcp_historico.assign(PERIODO=prcp_historico.FECHA.dt.to_period("M")).groupby("ID").PERIODO.agg(["min", "max"])
cobertura_mensual = cobertura_mensual.join(extremos, on="ID")
cobertura_mensual["en_actividad"] = (
    (cobertura_mensual.PERIODO >= cobertura_mensual["min"]) &
    (cobertura_mensual.PERIODO <= cobertura_mensual["max"])
)
display(cobertura_mensual.drop(columns=["min", "max"]).head(10))

,ID,PERIODO,dias_validos,dias_mes,cobertura,suficiente,ANIO,en_actividad
0,NZ000093292,2016-01,31,31,1.000000,True,2016,True
1,NZ000093292,2016-02,29,29,1.000000,True,2016,True
2,NZ000093292,2016-03,31,31,1.000000,True,2016,True
3,NZ000093292,2016-04,29,30,0.966667,True,2016,True
4,NZ000093292,2016-05,31,31,1.000000,True,2016,True
5,NZ000093292,2016-06,30,30,1.000000,True,2016,True
6,NZ000093292,2016-07,31,31,1.000000,True,2016,True
7,NZ000093292,2016-08,30,31,0.967742,True,2016,True
8,NZ000093292,2016-09,30,30,1.000000,True,2016,True
9,NZ000093292,2016-10,31,31,1.000000,True,2016,True


In [49]:
# Informe compacto de viabilidad del objetivo mensual
resumen_cobertura = cobertura_mensual.groupby("ANIO").agg(
    meses_evaluados=("suficiente", "size"),
    meses_suficientes=("suficiente", "sum"),
    cobertura_promedio=("cobertura", "mean")
)
resumen_cobertura["pct_meses_suficientes"] = (
    100 * resumen_cobertura.meses_suficientes / resumen_cobertura.meses_evaluados
)
print("Cobertura histórica por año (2016–2025):")
display(resumen_cobertura.round(2))

ranking = (
    cobertura_mensual.loc[cobertura_mensual.en_actividad]
    .groupby("ID")
    .agg(meses_activos=("suficiente", "size"),
         meses_suficientes=("suficiente", "sum"),
         cobertura_promedio=("cobertura", "mean"))
)
ranking["pct_meses_suficientes"] = (
    100 * ranking.meses_suficientes / ranking.meses_activos
)
ranking = ranking.sort_values(["meses_suficientes", "pct_meses_suficientes"], ascending=False)
print("Estaciones con más meses completos (solo período de actividad):")
display(ranking.head(15).round(2))

print("RESULTADO FINAL:")
print(f"Meses estación con cobertura ≥ {UMBRAL_COBERTURA:.0%}: "
      f"{int(cobertura_mensual.suficiente.sum())} de {len(cobertura_mensual)}")
print("Estaciones con al menos 12 meses suficientes:", int((ranking.meses_suficientes >= 12).sum()))
print("2026: registros disponibles, excluidos de la evaluación de completitud histórica.")

Cobertura histórica por año (2016–2025):


,meses_evaluados,meses_suficientes,cobertura_promedio,pct_meses_suficientes
ANIO,,,,
2016,156,134,0.90,85.90
2017,156,139,0.91,89.10
2018,156,123,0.87,78.85
2019,156,115,0.84,73.72
2020,156,128,0.85,82.05
2021,156,117,0.84,75.00
2022,156,135,0.88,86.54
2023,156,142,0.91,91.03
2024,156,116,0.88,74.36


Estaciones con más meses completos (solo período de actividad):


,meses_activos,meses_suficientes,cobertura_promedio,pct_meses_suficientes
ID,,,,
NZ000936150,116,109,0.98,93.97
NZM00093781,116,109,0.98,93.97
NZ000933090,116,107,0.98,92.24
NZ000093292,116,105,0.96,90.52
NZM00093110,117,102,0.91,87.18
NZ000093417,119,102,0.89,85.71
NZ000093844,111,99,0.95,89.19
NZM00093439,119,99,0.89,83.19
NZ000939450,120,98,0.88,81.67


RESULTADO FINAL:
Meses estación con cobertura ≥ 90%: 1183 de 1560
Estaciones con al menos 12 meses suficientes: 12
2026: registros disponibles, excluidos de la evaluación de completitud histórica.


## 9. Limpieza de auxiliares locales

Ejecutar al terminar el análisis. Se elimina únicamente la carpeta `datos_auxiliares_noaa` y sus metadatos descargados. No se modifican objetos en S3 ni archivos de la notebook.

In [50]:
ELIMINAR_CACHE = True

if ELIMINAR_CACHE:
    carpeta = CACHE.resolve()

    # Comprobar que se elimina únicamente la carpeta auxiliar prevista
    if carpeta.name != "datos_auxiliares_noaa":
        raise ValueError("La ruta no corresponde a la carpeta auxiliar.")

    if carpeta.is_dir():
        shutil.rmtree(carpeta)
        print(f"Carpeta eliminada: {carpeta}")
    else:
        print("La carpeta auxiliar no existe.")

Carpeta eliminada: D:\de_proyectos\moaa_ghcn_precipitaciones\repo\NOAA_precipitaciones\notebooks\datos_auxiliares_noaa
